In [13]:
# import bibliotek
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import time

In [2]:
# import danych
data_raw = pd.read_csv("australia.csv")
data_raw

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,17.9,35.2,0.0,12.0,12.3,48.0,6.0,20.0,20.0,13.0,1006.3,1004.4,2.0,5.0,26.6,33.4,0,0
1,18.4,28.9,0.0,14.8,13.0,37.0,19.0,19.0,30.0,8.0,1012.9,1012.1,1.0,1.0,20.3,27.0,0,0
2,19.4,37.6,0.0,10.8,10.6,46.0,30.0,15.0,42.0,22.0,1012.3,1009.2,1.0,6.0,28.7,34.9,0,0
3,21.9,38.4,0.0,11.4,12.2,31.0,6.0,6.0,37.0,22.0,1012.7,1009.1,1.0,5.0,29.1,35.6,0,0
4,24.2,41.0,0.0,11.2,8.4,35.0,17.0,13.0,19.0,15.0,1010.7,1007.4,1.0,6.0,33.6,37.6,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56415,19.3,33.4,0.0,6.0,11.0,35.0,9.0,20.0,63.0,32.0,1013.9,1010.5,0.0,1.0,24.5,32.3,0,0
56416,21.2,32.6,0.0,7.6,8.6,37.0,13.0,11.0,56.0,28.0,1014.6,1011.2,7.0,0.0,24.8,32.0,0,0
56417,20.7,32.8,0.0,5.6,11.0,33.0,17.0,11.0,46.0,23.0,1015.3,1011.8,0.0,0.0,24.8,32.1,0,0
56418,19.5,31.8,0.0,6.2,10.6,26.0,9.0,17.0,62.0,58.0,1014.9,1010.7,1.0,1.0,24.8,29.2,0,0


In [51]:
# funkcja licząca średnią dokładność, czas i liczbę wziętych kolumn
def model_results(data_raw, pca_percent_of_variance, tests_n):
    
    # przeskalowanie danych
    X_train, X_test, Y_train, Y_test = train_test_split(data_raw.drop('RainTomorrow', axis=1), data_raw['RainTomorrow'])
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # bez pca
    if pca_percent_of_variance == 1:      
        X_train_pca = X_train
        X_test_pca = X_test
        columns_n = X_train.shape[1]
    else:
        # wykonanie PCA
        pca = PCA(pca_percent_of_variance)
        pca.fit(X_train)
        X_train_pca = pca.transform(X_train)
        X_test_pca = pca.transform(X_test)
        columns_n = pca.n_components_
    
    sum_time = 0;
    sum_score = 0;
    
    for i in range(tests_n):
        
        # zaczynamy mierzyć czas
        start_time = time.process_time()

        # stworzenie modelu - użyjemy regresji logistycznej
        logisticRegr = LogisticRegression(solver = 'lbfgs')
        logisticRegr.fit(X_train_pca, Y_train)
        score = logisticRegr.score(X_test_pca, Y_test)
            
        end_time = time.process_time()
        
        sum_score += score
        sum_time += end_time - start_time

    
    return (sum_score / tests_n, sum_time / tests_n, columns_n)

In [53]:
results = []
results.append(["Accuracy modelu", "czas (sekundy)", "liczba wzietych kolumn"])

# wykonujemy uczenie dla podanego % wyjasnianej wariancji w pca
for pca_percent in [1, 0.999, 0.99, 0.95, 0.9, 0.85]:
    
    # robimy 20 testów i liczymy średnią
    results.append(list(model_results(data_raw, pca_percent, 20)))

# robienie  ładnej tabelki
table = pd.DataFrame.from_records(results)

# naprawa headera tabelki
new_header = table.iloc[0] #grab the first row for the header
table = table[1:] #take the data less the header row
table.columns = new_header #set the header row as the df header

table

,Accuracy modelu,czas (sekundy),liczba wzietych kolumn
1,0.857923,0.59216,17
2,0.853456,0.434892,16
3,0.84509,0.225844,13
4,0.843956,0.183321,11
5,0.842042,0.185454,9
6,0.835732,0.154816,7


## Komentarz do wyników
Jak widzimy, ograniczając liczbę kolumn w danych udało się zaoszczędzić sporo czasu, nie tracąc jednocześnie dużo na skuteczności rozwiązania. Wzięte przez nas dane są względnie małe objętościowo, ale gdybyśmy mieli do czynienia z setkami kolumn i setkami tysięcy wierszy, to PCA pozwoliło by nam znacząco przyspieszyć uczenie modelu.